# Data preparation

**Goal:** load IBTrACS with UTC timestamps, remove short tracks, and put every
storm in chronological order. This notebook is an inspection guide; the
production implementation lives in `storm_tracks.data`.

In [ ]:
from pathlib import Path
import tomllib

from storm_tracks.data import load_ibtracs, prepare_tracks

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
with (PROJECT_ROOT / "configs" / "pipeline.toml").open("rb") as file:
    config = tomllib.load(file)
raw_path = PROJECT_ROOT / config["paths"]["raw_data"]
intermediate_path = PROJECT_ROOT / "data" / "hurricane_paths_prepared.pkl"

## Load and normalize the two-header source file

In [ ]:
raw_tracks = load_ibtracs(raw_path)
assert str(raw_tracks["iso_time"].dtype) == "datetime64[ns, UTC]"
raw_tracks[["sid", "iso_time", "lat", "lon"]].head()

## Filter and order complete trajectories

In [ ]:
prepared_tracks = prepare_tracks(
    raw_tracks,
    min_observations=config["preparation"]["min_observations"],
)
assert prepared_tracks.groupby("sid")["iso_time"].apply(
    lambda values: values.is_monotonic_increasing
).all()
intermediate_path.parent.mkdir(parents=True, exist_ok=True)
prepared_tracks.to_pickle(intermediate_path)
print(f"Saved the granular handoff to {intermediate_path}")

## Next step

Continue to `02_feature_engineering.ipynb`, which reads the saved granular
handoff. Use `scripts/prepare_data.py` to combine both preparation steps in
one canonical command.